In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle, os, time, warnings
warnings.filterwarnings('ignore')
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
import matplotlib.gridspec as gridspec
from lifelines import CoxPHFitter
from lifelines.utils import concordance_index
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.metrics import cumulative_dynamic_auc
from scipy.stats import spearmanr

import shap


CACHE_DIR    = '/tmp/tnbc_M13'
os.makedirs(CACHE_DIR, exist_ok=True)
RANDOM_STATE = 42

# B=10 -> quick test (~2 min).  B=500 -> final reporting run (~15 min).
B_OOB      = 200    # number of bootstrap resamples (quick test: 50, final report: 300-500)
B_SEED     = 100    # bootstrap resamples per seed (unused by the cells kept in this notebook)


# top-K for forward greedy selection (unused by the cells kept in this notebook)
K_TOP = 5

print('Setup OK')
print(f'B_OOB : {B_OOB}')

Setup OK
B_OOB : 200


In [2]:
spatial_marker_df = pd.read_csv("data/targeted_proteomics/target_final_pep_with_scores3.tsv", encoding='utf-8-sig', sep = '\t')
pr = pd.read_csv("data/targeted_proteomics/report.pr_matrix.tsv", sep = '\t')
cc = pd.read_csv("data/targeted_proteomics/column_check.txt", sep = '\t')

spatial_marker_selected = spatial_marker_df[spatial_marker_df.Precursor_selected]
pr   = pr[pr["Precursor.Id"].isin(spatial_marker_selected["Precursor.Id"].tolist())]
pr = pr.rename(columns={row[0] : row[1] for row in cc[["pr_col","SampleID"]].values}).copy()

meta_cols = ['Protein.Group','Protein.Ids','Protein.Names','Genes','First.Protein.Description',
             'Proteotypic','Stripped.Sequence','Modified.Sequence','Precursor.Charge','Precursor.Id']


sample_cols = [c for c in pr.columns if c not in meta_cols]


In [3]:
cc_idx = cc.set_index('SampleID').loc[sample_cols]

y_time  = cc_idx['RFS'].values.astype(float)
y_event = cc_idx['Recur'].values.astype(int)
risk    = cc_idx['risk_scores'].values.astype(float)
n       = len(y_time)

# 0→NaN, log2, median-impute, z-score
X_raw = pr.set_index('Precursor.Id')[sample_cols].T.replace(0, np.nan)
X_log = np.log2(X_raw)
X_imp = X_log.fillna(X_log.median())
X_z   = (X_imp - X_imp.mean()) / X_imp.std()

all_pids = list(X_z.columns)
gene_map = dict(zip(pr['Precursor.Id'].astype(str), pr['Genes'].astype(str)))
seq_map  = dict(zip(pr['Precursor.Id'].astype(str), pr['Stripped.Sequence'].astype(str)))
chg_map  = dict(zip(pr['Precursor.Id'].astype(str), pr['Precursor.Charge'].astype(str)))
gene_to_idx = {}
for i, pid in enumerate(all_pids):
    gene_to_idx.setdefault(gene_map[pid], []).append(i)

def surv(t, e):
    return Surv.from_arrays(event=e.astype(bool), time=t)
y_surv_all = surv(y_time, y_event)
times_e    = np.array([730, 1095, 1825])

print(f'n={n}, events={y_event.sum()} ({y_event.mean()*100:.1f}%)')
print(f'Precursors={len(all_pids)}, Genes={len(gene_to_idx)}')
print(f'RFS median={np.median(y_time):.0f}d ({np.median(y_time)/365:.1f}yr)')

n=96, events=12 (12.5%)
Precursors=33, Genes=13
RFS median=2270d (6.2yr)


## Session 2 — Precursor selection (one per gene)

**Criterion: inter-peptide correlation (IPC)**

When a gene has multiple peptides, pick the one with the **highest mean|Pearson r| against the other peptides of that gene**.

- Outcome is never used -> no leakage
- The most internally consistent peptide is treated as the most reliable representative
- Flag a measurement-quality warning when IPC < 0.35


In [4]:
def best_peptide_by_ipc(gene):
    """
    Returns the peptide with the highest inter-peptide correlation (IPC) within a gene.
    IPC = mean|Pearson r| against the other peptides of the same gene.
    Outcome is not used.
    """
    pids_g = [all_pids[i] for i in gene_to_idx[gene]]
    if len(pids_g) == 1:
        return pids_g[0], np.nan

    mean_corr = {}
    for p in pids_g:
        others = [q for q in pids_g if q != p]
        x = X_z[p].values
        rs = []
        for q in others:
            mask = ~(np.isnan(x) | np.isnan(X_z[q].values))
            if mask.sum() >= 5:
                rs.append(abs(np.corrcoef(x[mask], X_z[q].values[mask])[0, 1]))
        mean_corr[p] = np.mean(rs) if rs else 0.0

    best = max(mean_corr, key=mean_corr.get)
    return best, mean_corr[best]

BP  = {}   # gene → best precursor.id
IPC = {}   # gene → mean|r|

print(f'{"Gene":10s}  {"Best Precursor.Id":35s}  {"Stripped Seq":25s}  {"Chg":>3s}  {"IPC":>6s}  n_pep')
print('-' * 90)
for gene in sorted(gene_to_idx):
    pid, ipc = best_peptide_by_ipc(gene)
    BP[gene]  = pid
    IPC[gene] = ipc
    ipc_str   = f'{ipc:.3f}' if not np.isnan(ipc) else '  N/A'
    warn      = ' ⚠' if (not np.isnan(ipc) and ipc < 0.35) else ''
    n_pep     = len(gene_to_idx[gene])
    print(f'  {gene:10s}  {pid:35s}  {seq_map[pid]:25s}  +{chg_map[pid]}  {ipc_str:>6s}  {n_pep}{warn}')

def gz(g): return X_z[BP[g]].values
def zs(x): return (x - x.mean()) / (x.std() + 1e-9)
def rnk(x): return pd.Series(x).rank(ascending=True).values

Gene        Best Precursor.Id                    Stripped Seq               Chg     IPC  n_pep
------------------------------------------------------------------------------------------
  ARHGDIB     TLLGDGPVVTDPK2                       TLLGDGPVVTDPK              +2   0.585  3
  EPPK1       TTVPQLLASVQR2                        TTVPQLLASVQR               +2   0.674  3
  GET3        ESVLIISTDPAHNISDAFDQK3               ESVLIISTDPAHNISDAFDQK      +3   0.469  3
  LAP3        TIQVDNTDAEGR2                        TIQVDNTDAEGR               +2   0.795  3
  LCP1        NEALIALLR2                           NEALIALLR                  +2   0.691  3
  MYH11       QLEEAEEESQR2                         QLEEAEEESQR                +2     N/A  1
  P4HA1       FHDIISDAEIEIVK3                      FHDIISDAEIEIVK             +3   0.806  3
  PSME2       KQVEVFR2                             KQVEVFR                    +2   0.316  3 ⚠
  RPS15       EAPPMEKPEVVK3                        EAPPMEKPEVVK             

## Session 3 — DEG direction, defined from the spatial proteomics experiment

### Experiment B (P1 ∩ P2 — DEGs concordant across both patients)
Direction is defined from the within-patient spatial proteomics comparison of high-risk vs. low-risk regions.
The following **13 genes are used as-is**, with no additional screening/intersection step.

```
UP   (higher in high-risk regions, 5):  P4HA1, EPPK1, RPS15, CNN3, NDRG1
DOWN (lower in high-risk regions, 8):  GET3, LCP1, LAP3, GBP1, SERPINB9,
                                     ARHGDIB, PSME2, SAMHD1
```

In [5]:
# ---- final gene set (13 genes, used as-is, no intersection step) ----
UP_GENES   = [  'EPPK1', 'MYH11', 'P4HA1', 'RPS15', 'YAP1']                  # up in high-risk regions
DOWN_GENES = ['ARHGDIB', 'GET3', 'LAP3', 'LCP1', 'PSME2', 'SERPINB9', 'TYMP', 'WAS']                # down in high-risk regions

# a gene must have a selected peptide in the panel (BP) to enter the composite -> check for that
UP      = [g for g in UP_GENES   if g in BP]
DOWN    = [g for g in DOWN_GENES if g in BP]
missing = [g for g in UP_GENES + DOWN_GENES if g not in BP]

print('=== Final gene set ===')
print(f'UP   ({len(UP)}/{len(UP_GENES)}): {UP}')
print(f'DOWN ({len(DOWN)}/{len(DOWN_GENES)}): {DOWN}')
print(f'genes actually used in the composite: {len(UP) + len(DOWN)}')
if missing:
    print(f'genes missing from the panel (BP) ({len(missing)}): {missing}')
else:
    print('All 13 genes are present in the panel.')

print('\n=== Direction mapping ===')
print(f'{"Gene":10s}  {"Direction":15s}')
print('-'*28)
for g in DOWN + UP:
    d = 'low=bad  (\u2193)' if g in DOWN else 'high=bad (\u2191)'
    print(f'  {g:10s}  {d:15s}')

=== Final gene set ===
UP   (5/5): ['EPPK1', 'MYH11', 'P4HA1', 'RPS15', 'YAP1']
DOWN (8/8): ['ARHGDIB', 'GET3', 'LAP3', 'LCP1', 'PSME2', 'SERPINB9', 'TYMP', 'WAS']
genes actually used in the composite: 13
All 13 genes are present in the panel.

=== Direction mapping ===
Gene        Direction      
----------------------------
  ARHGDIB     low=bad  (↓)   
  GET3        low=bad  (↓)   
  LAP3        low=bad  (↓)   
  LCP1        low=bad  (↓)   
  PSME2       low=bad  (↓)   
  SERPINB9    low=bad  (↓)   
  TYMP        low=bad  (↓)   
  WAS         low=bad  (↓)   
  EPPK1       high=bad (↑)   
  MYH11       high=bad (↑)   
  P4HA1       high=bad (↑)   
  RPS15       high=bad (↑)   
  YAP1        high=bad (↑)   


## Session 4 — Composite score calculation

In [6]:
def make_comp(down_genes, up_genes):
    """composite = -mean(z of DOWN) + mean(z of UP)"""
    score = np.zeros(n)
    d = [g for g in down_genes if g in BP]
    u = [g for g in up_genes   if g in BP]
    if d: score -= np.column_stack([gz(g) for g in d]).mean(axis=1)
    if u: score += np.column_stack([gz(g) for g in u]).mean(axis=1)
    return score

# ---- composite definition (13 genes, no intersection step) ----
FINAL_DOWN = DOWN
FINAL_UP   = UP
N_GENES    = len(FINAL_DOWN) + len(FINAL_UP)

comp_final = make_comp(FINAL_DOWN, FINAL_UP)
SE_final   = rnk(risk) + rnk(comp_final)            # = M8 (composite + risk)

print('=== Composite score summary statistics ===')
r_ev, p_ev = spearmanr(comp_final, y_event)
r_rs, _    = spearmanr(comp_final, risk)
print(f'  composite({N_GENES}g)  mean={comp_final.mean():+.3f}  std={comp_final.std():.3f}  '
      f'r(event)={r_ev:+.3f}(p={p_ev:.3f})  r(risk)={r_rs:+.3f}')

# LOO M1 (Cox PH -- risk only, comparison baseline)
def loo_m1():
    p = np.zeros(n)
    for i in range(n):
        tr = [j for j in range(n) if j != i]
        df = pd.DataFrame({'time':y_time[tr],'event':y_event[tr],'risk':risk[tr]})
        cph = CoxPHFitter(penalizer=0.01); cph.fit(df,'time','event')
        p[i] = risk[i] * cph.params_['risk']
    return p

def ev(score):
    c  = concordance_index(y_time, -score, y_event)
    a, ia = cumulative_dynamic_auc(y_surv_all, y_surv_all, score, times_e)
    return {'c':c, 'a2':a[0], 'a3':a[1], 'a5':a[2], 'ia':ia}

print('\nM1 LOO...')
t0 = time.time()
p_m1 = loo_m1()
m1   = ev(p_m1)
c_m1 = m1['c']
print(f'  M1: C={c_m1:.4f}  ({time.time()-t0:.0f}s)')

# ---- the 3 model scores being compared ----
MODELS = {
    'M1 (risk only)':                    p_m1,
    f'M2 (composite {N_GENES}g)':        rnk(comp_final),
    f'M8 (composite {N_GENES}g + risk)': SE_final,
}

=== Composite score summary statistics ===
  composite(13g)  mean=+0.000  std=0.824  r(event)=+0.123(p=0.233)  r(risk)=+0.119

M1 LOO...


  M1: C=0.7104  (1s)


## Session 5 — Performance evaluation (LOO / OOB)

In [7]:
# ---- LOO model comparison (M1 / M2 / M8) ----
print('LOO comparison...')
models_loo = {}
for name, sc in MODELS.items():
    models_loo[name] = ev(sc)

print(f'\n  {"Model":34s}  {"C":>7}  {"dC":>7}  {"2y":>7}  {"3y":>7}  {"5y":>7}  {"iAUC":>7}')
print('  '+'-'*80)
for name, r in models_loo.items():
    dc = r['c'] - c_m1
    flag = '★' if dc > 0.03 else ' '
    print(f' {flag} {name:34s}  {r["c"]:7.4f}  {dc:>+7.4f}  '
          f'{r["a2"]:7.4f}  {r["a3"]:7.4f}  {r["a5"]:7.4f}  {r["ia"]:7.4f}')

LOO comparison...

  Model                                     C       dC       2y       3y       5y     iAUC
  --------------------------------------------------------------------------------
   M1 (risk only)                       0.7104  +0.0000   0.7665   0.7643   0.7131   0.7557
   M2 (composite 13g)                   0.6290  -0.0814   0.5743   0.6515   0.6700   0.6065
   M8 (composite 13g + risk)            0.7331  +0.0227   0.7488   0.7841   0.7617   0.7575


In [8]:
# ---- OOB bootstrap: same model comparison as LOO ----

def run_oob(score, B=200, seed=42):
    rng = np.random.RandomState(seed)
    dcs, dias = [], []
    auc_store = {730: [], 1095: [], 1825: []}

    for _ in range(B):
        tr  = rng.choice(n, n, replace=True)
        oob = np.setdiff1d(np.arange(n), tr)

        if y_event[tr].sum() < 2 or y_event[oob].sum() < 1 or len(oob) < 5:
            continue

        oob_max = y_time[oob].max()
        valid   = times_e[times_e < oob_max]

        if len(valid) == 0:
            continue

        try:
            # baseline: M1 risk-only Cox
            df_ = pd.DataFrame({
                'time': y_time[tr],
                'event': y_event[tr],
                'risk': risk[tr]
            })

            cph = CoxPHFitter(penalizer=0.01)
            cph.fit(df_, 'time', 'event')

            p1 = risk[oob] * cph.params_['risk']

            if np.std(p1) < 1e-12:
                continue

            c1 = concordance_index(y_time[oob], -p1, y_event[oob])

            # comparison model
            pp = score[oob]

            if np.std(pp) < 1e-12:
                continue

            cp = concordance_index(y_time[oob], -pp, y_event[oob])
            dcs.append(cp - c1)

            a1_, ia1_ = cumulative_dynamic_auc(
                surv(y_time[tr], y_event[tr]),
                surv(y_time[oob], y_event[oob]),
                p1,
                valid
            )

            ap_, iap_ = cumulative_dynamic_auc(
                surv(y_time[tr], y_event[tr]),
                surv(y_time[oob], y_event[oob]),
                pp,
                valid
            )

            dias.append(iap_ - ia1_)

            for tt, v1, vp in zip(valid, a1_, ap_):
                if np.isfinite(v1) and np.isfinite(vp):
                    auc_store[tt].append(vp - v1)

        except Exception:
            continue

    d = np.array(dcs)

    return {
        'dc':      d.mean() if len(d) else np.nan,
        'dc_lo':   np.percentile(d, 2.5) if len(d) else np.nan,
        'dc_hi':   np.percentile(d, 97.5) if len(d) else np.nan,
        'p_gt':    np.mean(d > 0) if len(d) else np.nan,
        'dia':     np.mean(dias) if dias else np.nan,
        'dauc_2y': np.mean(auc_store[730]) if auc_store[730] else np.nan,
        'dauc_3y': np.mean(auc_store[1095]) if auc_store[1095] else np.nan,
        'dauc_5y': np.mean(auc_store[1825]) if auc_store[1825] else np.nan,
        'n':       len(d)
    }


print(f'OOB comparison (B={B_OOB})...')
t0 = time.time()

refs_oob = {
    f'M2 (composite {N_GENES}g)':        rnk(comp_final),
    f'M8 (composite {N_GENES}g + risk)': SE_final,
}

models_oob = {}
for name, sc in refs_oob.items():
    models_oob[name] = run_oob(sc, B=B_OOB)

print(f'  done ({time.time()-t0:.0f}s)')

print()
print(f'  {"Model":34s}  {"OOB dC":>8}  {"95%CI":>17}  {"P(>0)":>6}  {"diAUC":>7}  {"d2y":>7}  {"d3y":>7}  {"d5y":>7}  {"n":>5}')
print('  ' + '-'*108)
for name, o in models_oob.items():
    flag = '★' if o['dc'] > 0.03 else ' '
    print(
        f' {flag} {name:34s}  '
        f'{o["dc"]:>+8.4f}  '
        f'[{o["dc_lo"]:+.3f},{o["dc_hi"]:+.3f}]  '
        f'{o["p_gt"]:>5.0%}  '
        f'{o["dia"]:>+7.4f}  '
        f'{o["dauc_2y"]:>+7.4f}  '
        f'{o["dauc_3y"]:>+7.4f}  '
        f'{o["dauc_5y"]:>+7.4f}  '
        f'{o["n"]:5d}'
    )

OOB comparison (B=200)...


  done (7s)

  Model                                 OOB dC              95%CI   P(>0)    diAUC      d2y      d3y      d5y      n
  ------------------------------------------------------------------------------------------------------------
   M2 (composite 13g)                   -0.0357  [-0.324,+0.414]    33%     +nan  -0.1712  -0.0818  +0.0081    199
 ★ M8 (composite 13g + risk)            +0.0599  [-0.131,+0.534]    69%     +nan  +0.0008  +0.0538  +0.0952    199


In [9]:
# ============================================================
# run_oob_abs: OOB bootstrap returning ABSOLUTE metrics
# (baseline M1 risk-only Cox  +  model)  plus deltas.
# Same resampling scheme as run_oob(); used by the summary table below.
# ============================================================

def run_oob_abs(score, B=200, seed=42):
    rng = np.random.RandomState(seed)

    base_c, base_ia = [], []
    mdl_c,  mdl_ia  = [], []
    dcs,    dias    = [], []
    base_auc = {730: [], 1095: [], 1825: []}
    mdl_auc  = {730: [], 1095: [], 1825: []}

    for _ in range(B):
        tr  = rng.choice(n, n, replace=True)
        oob = np.setdiff1d(np.arange(n), tr)

        if y_event[tr].sum() < 2 or y_event[oob].sum() < 1 or len(oob) < 5:
            continue

        oob_max = y_time[oob].max()
        valid   = times_e[times_e < oob_max]
        if len(valid) == 0:
            continue

        try:
            # baseline: M1 risk-only Cox
            df_ = pd.DataFrame({
                'time': y_time[tr],
                'event': y_event[tr],
                'risk': risk[tr]
            })
            cph = CoxPHFitter(penalizer=0.01)
            cph.fit(df_, 'time', 'event')

            p1 = risk[oob] * cph.params_['risk']
            if np.std(p1) < 1e-12:
                continue
            c1 = concordance_index(y_time[oob], -p1, y_event[oob])

            # comparison model
            pp = score[oob]
            if np.std(pp) < 1e-12:
                continue
            cp = concordance_index(y_time[oob], -pp, y_event[oob])

            a1_, ia1_ = cumulative_dynamic_auc(
                surv(y_time[tr], y_event[tr]),
                surv(y_time[oob], y_event[oob]),
                p1,
                valid
            )
            ap_, iap_ = cumulative_dynamic_auc(
                surv(y_time[tr], y_event[tr]),
                surv(y_time[oob], y_event[oob]),
                pp,
                valid
            )

            base_c.append(c1);  mdl_c.append(cp);   dcs.append(cp - c1)
            base_ia.append(ia1_); mdl_ia.append(iap_); dias.append(iap_ - ia1_)

            for tt, v1, vp in zip(valid, a1_, ap_):
                if np.isfinite(v1):
                    base_auc[tt].append(v1)
                if np.isfinite(vp):
                    mdl_auc[tt].append(vp)

        except Exception:
            continue

    def mean_(x): return float(np.mean(x)) if len(x) else np.nan
    def lo_(x):   return float(np.percentile(x, 2.5))  if len(x) else np.nan
    def hi_(x):   return float(np.percentile(x, 97.5)) if len(x) else np.nan

    d = np.array(dcs)

    return {
        # baseline (M1 risk-only) absolute
        'base_c':       mean_(base_c),
        'base_c_lo':    lo_(base_c),
        'base_c_hi':    hi_(base_c),
        'base_iAUC':    mean_(base_ia),
        'base_iAUC_lo': lo_(base_ia),
        'base_iAUC_hi': hi_(base_ia),
        'base_AUC_2y':  mean_(base_auc[730]),
        'base_AUC_3y':  mean_(base_auc[1095]),
        'base_AUC_5y':  mean_(base_auc[1825]),

        # model absolute
        'model_c':       mean_(mdl_c),
        'model_c_lo':    lo_(mdl_c),
        'model_c_hi':    hi_(mdl_c),
        'model_iAUC':    mean_(mdl_ia),
        'model_iAUC_lo': lo_(mdl_ia),
        'model_iAUC_hi': hi_(mdl_ia),
        'model_AUC_2y':  mean_(mdl_auc[730]),
        'model_AUC_3y':  mean_(mdl_auc[1095]),
        'model_AUC_5y':  mean_(mdl_auc[1825]),

        # deltas (model − baseline)
        'dc':   mean_(dcs),
        'dia':  mean_(dias),
        'p_gt': float(np.mean(d > 0)) if len(d) else np.nan,
        'n':    len(dcs),
    }

In [10]:
# ============================================================
# Run OOB absolute model evaluation  (M2 / M8; M1 = baseline)
# ============================================================
import time

print(f'OOB absolute + delta comparison (B={B_OOB})...')
t0 = time.time()

refs_oob = {
    f'M2 (composite {N_GENES}g)':        rnk(comp_final),
    f'M8 (composite {N_GENES}g + risk)': SE_final,
}

models_oob_abs = {}
for name, sc in refs_oob.items():
    print(f'  running: {name}')
    models_oob_abs[name] = run_oob_abs(sc, B=B_OOB)

print(f'  done ({time.time()-t0:.0f}s)')
print(f'  evaluated models: {len(models_oob_abs)}')

OOB absolute + delta comparison (B=200)...
  running: M2 (composite 13g)


  running: M8 (composite 13g + risk)


  done (7s)
  evaluated models: 2


In [11]:
rows = []

# M1 baseline (recomputed identically inside each model's OOB loop; expected value is the same)
first_key = list(models_oob_abs.keys())[0]
b = models_oob_abs[first_key]
rows.append({
    'Model': 'M1 (H&E AI risk only)',
    'Components': 'H&E AI risk',
    'No. markers': 0,
    'Integration': 'Cox risk score',
    'OOB C-index': b['base_c'],
    'C-index 95% CI': f"[{b['base_c_lo']:.3f}, {b['base_c_hi']:.3f}]",
    'OOB iAUC': b['base_iAUC'],
    'iAUC 95% CI': f"[{b['base_iAUC_lo']:.3f}, {b['base_iAUC_hi']:.3f}]",
    'AUC@2y': b['base_AUC_2y'],
    'AUC@3y': b['base_AUC_3y'],
    'AUC@5y': b['base_AUC_5y'],
    'ΔC-index vs M1': np.nan,
    'ΔiAUC vs M1': np.nan,
    'P(ΔC>0)': np.nan,
    'n bootstrap': b['n'],
})

for name, o in models_oob_abs.items():
    if name.startswith('M2'):
        components  = f'composite peptide score ({N_GENES} genes)'
        n_markers   = N_GENES
        integration = 'Marker score only'
    elif name.startswith('M8'):
        components  = f'H&E AI risk + composite ({N_GENES} genes)'
        n_markers   = N_GENES
        integration = 'Rank-sum'
    else:
        components  = ''
        n_markers   = np.nan
        integration = ''

    rows.append({
        'Model': name,
        'Components': components,
        'No. markers': n_markers,
        'Integration': integration,
        'OOB C-index': o['model_c'],
        'C-index 95% CI': f"[{o['model_c_lo']:.3f}, {o['model_c_hi']:.3f}]",
        'OOB iAUC': o['model_iAUC'],
        'iAUC 95% CI': f"[{o['model_iAUC_lo']:.3f}, {o['model_iAUC_hi']:.3f}]",
        'AUC@2y': o['model_AUC_2y'],
        'AUC@3y': o['model_AUC_3y'],
        'AUC@5y': o['model_AUC_5y'],
        'ΔC-index vs M1': o['dc'],
        'ΔiAUC vs M1': o['dia'],
        'P(ΔC>0)': o['p_gt'],
        'n bootstrap': o['n'],
    })

oob_table = pd.DataFrame(rows)

display_cols = [
    'Model', 'Components', 'No. markers', 'Integration',
    'OOB C-index', 'C-index 95% CI',
    'OOB iAUC', 'iAUC 95% CI',
    'AUC@2y', 'AUC@3y', 'AUC@5y',
    'ΔC-index vs M1', 'ΔiAUC vs M1', 'P(ΔC>0)', 'n bootstrap'
]

display(
    oob_table[display_cols]
    .style.format({
        'OOB C-index': '{:.3f}',
        'OOB iAUC': '{:.3f}',
        'AUC@2y': '{:.3f}',
        'AUC@3y': '{:.3f}',
        'AUC@5y': '{:.3f}',
        'ΔC-index vs M1': '{:+.3f}',
        'ΔiAUC vs M1': '{:+.3f}',
        'P(ΔC>0)': '{:.0%}',
    })
)

oob_table.to_csv('OOB_absolute_model_performance_table.csv', index=False)
print('Saved: OOB_absolute_model_performance_table.csv')

,Model,Components,No. markers,Integration,OOB C-index,C-index 95% CI,OOB iAUC,iAUC 95% CI,AUC@2y,AUC@3y,AUC@5y,ΔC-index vs M1,ΔiAUC vs M1,P(ΔC>0),n bootstrap
0,M1 (H&E AI risk only),H&E AI risk,0,Cox risk score,0.679,"[0.274, 0.875]",nan,"[nan, nan]",0.757,0.741,0.675,+nan,+nan,nan%,199
1,M2 (composite 13g),composite peptide score (13 genes),13,Marker score only,0.643,"[0.427, 0.843]",nan,"[nan, nan]",0.585,0.659,0.683,-0.036,+nan,33%,199
2,M8 (composite 13g + risk),H&E AI risk + composite (13 genes),13,Rank-sum,0.739,"[0.531, 0.898]",nan,"[nan, nan]",0.757,0.795,0.770,+0.060,+nan,69%,199


Saved: OOB_absolute_model_performance_table.csv


## Session 6 — Cox PH analysis & Kaplan-Meier

In [12]:
# ============================================================
# Direction-aligned univariate Cox PH
# HR is reported per 1 SD increase in the risk-aligned direction
#
# FINAL_UP:
#   higher expression = higher-risk direction
#
# FINAL_DOWN:
#   lower expression = higher-risk direction
#   -> multiply standardized expression by -1
# ============================================================

import numpy as np
import pandas as pd

from lifelines import CoxPHFitter


# ---- z-score helper ----
def zs(x):
    x = np.asarray(x, dtype=float)

    mean_x = np.nanmean(x)
    std_x = np.nanstd(x, ddof=0)

    if std_x == 0 or not np.isfinite(std_x):
        return np.zeros_like(x, dtype=float)

    return (x - mean_x) / std_x


# ── Overall scores: higher score = higher risk ─────────────────
risk_z = zs(risk)
comp_z = zs(comp_final)
SE_score_z = zs(SE_final)


# ── 1. Direction-aligned univariate Cox ────────────────────────
print('═' * 95)
print('  Univariate Cox PH — direction-aligned, per 1 SD')
print('═' * 95)

print(
    f'  {"Variable":30s}'
    f'  {"Direction":>15s}'
    f'  {"HR":>7s}'
    f'  {"95%CI_lo":>10s}'
    f'  {"95%CI_hi":>10s}'
    f'  {"p":>9s}'
    f'  {"C-index":>8s}'
)

print('  ' + '-' * 93)


# ------------------------------------------------------------
# assemble variables
# ------------------------------------------------------------
uni_vars = {
    'H&E risk score': {
        'x': risk_z,
        'direction': 'higher = risk'
    },

    'COMPOSITE': {
        'x': comp_z,
        'direction': 'higher = risk'
    },

    'H&E + COMPOSITE': {
        'x': SE_score_z,
        'direction': 'higher = risk'
    }
}


# ------------------------------------------------------------
# Individual proteins
# ------------------------------------------------------------
for g in FINAL_DOWN + FINAL_UP:

    expression_z = zs(gz(g))

    if g in FINAL_DOWN:
        # for this marker, lower expression means higher risk
        # flip the sign so that, after direction alignment, higher = higher risk
        aligned_x = -expression_z
        label = f'{g} (↓)'
        direction = 'lower = risk'

    else:
        # for this marker, higher expression means higher risk
        aligned_x = expression_z
        label = f'{g} (↑)'
        direction = 'higher = risk'

    uni_vars[label] = {
        'x': aligned_x,
        'direction': direction
    }


# ------------------------------------------------------------
# Cox fitting
# ------------------------------------------------------------
uni_results = {}

for name, info in uni_vars.items():

    x = np.asarray(info['x'], dtype=float)
    direction = info['direction']

    df = pd.DataFrame({
        'T': np.asarray(y_time, dtype=float),
        'E': np.asarray(y_event, dtype=int),
        'x': x
    })

    # drop missing/inf values
    df = df.replace([np.inf, -np.inf], np.nan).dropna()

    try:
        cph = CoxPHFitter(penalizer=0.01)

        cph.fit(
            df,
            duration_col='T',
            event_col='E',
            show_progress=False
        )

        beta = cph.params_['x']

        hr = np.exp(beta)

        lo = np.exp(
            cph.confidence_intervals_.loc[
                'x',
                '95% lower-bound'
            ]
        )

        hi = np.exp(
            cph.confidence_intervals_.loc[
                'x',
                '95% upper-bound'
            ]
        )

        p = cph.summary.loc['x', 'p']
        c = cph.concordance_index_

        uni_results[name] = {
            'Variable': name,
            'Direction': direction,
            'Coefficient': beta,
            'HR': hr,
            '95% CI lower': lo,
            '95% CI upper': hi,
            'p': p,
            'C-index': c,
            'n': len(df),
            'Events': int(df['E'].sum())
        }

        sig = '★' if p < 0.05 else ('†' if p < 0.10 else ' ')

        print(
            f' {sig} {name:30s}'
            f'  {direction:>15s}'
            f'  {hr:7.3f}'
            f'  {lo:10.3f}'
            f'  {hi:10.3f}'
            f'  {p:9.4f}'
            f'  {c:8.4f}'
        )

    except Exception as e:
        print(f'   {name:30s} ERROR: {e}')


print()
print('  ★ p < 0.05')
print('  † p < 0.10')
print()
print('  Direction-aligned interpretation:')
print('  • ↑ marker: HR per 1 SD higher expression')
print('  • ↓ marker: HR per 1 SD lower expression')
print('  • Therefore, HR > 1 indicates that the predefined direction holds.')


# ------------------------------------------------------------
# results table
# ------------------------------------------------------------
uni_results_df = pd.DataFrame(
    uni_results.values()
)

display(
    uni_results_df.style.format({
        'Coefficient': '{:.3f}',
        'HR': '{:.3f}',
        '95% CI lower': '{:.3f}',
        '95% CI upper': '{:.3f}',
        'p': '{:.4f}',
        'C-index': '{:.4f}'
    })
)

═══════════════════════════════════════════════════════════════════════════════════════════════
  Univariate Cox PH — direction-aligned, per 1 SD
═══════════════════════════════════════════════════════════════════════════════════════════════
  Variable                              Direction       HR    95%CI_lo    95%CI_hi          p   C-index
  ---------------------------------------------------------------------------------------------
 ★ H&E risk score                    higher = risk    1.717       1.046       2.820     0.0326    0.7040
   COMPOSITE                         higher = risk    1.510       0.836       2.728     0.1716    0.6290
 ★ H&E + COMPOSITE                   higher = risk    2.220       1.199       4.110     0.0112    0.7331
   ARHGDIB (↓)                        lower = risk    1.466       0.849       2.534     0.1702    0.5899
   GET3 (↓)                           lower = risk    1.601       0.873       2.938     0.1284    0.6427


   LAP3 (↓)                           lower = risk    1.484       0.806       2.733     0.2050    0.6015
 ★ LCP1 (↓)                           lower = risk    1.800       1.061       3.052     0.0292    0.6956
   PSME2 (↓)                          lower = risk    1.291       0.813       2.050     0.2784    0.6374


   SERPINB9 (↓)                       lower = risk    1.608       0.880       2.939     0.1227    0.6131


   TYMP (↓)                           lower = risk    1.109       0.638       1.929     0.7141    0.5180
   WAS (↓)                            lower = risk    1.250       0.743       2.100     0.4006    0.5973
   EPPK1 (↑)                         higher = risk    0.854       0.496       1.468     0.5672    0.5375
   MYH11 (↑)                         higher = risk    1.233       0.705       2.154     0.4625    0.5555
   P4HA1 (↑)                         higher = risk    1.147       0.654       2.011     0.6328    0.5677
   RPS15 (↑)                         higher = risk    0.772       0.449       1.325     0.3475    0.5983
   YAP1 (↑)                          higher = risk    0.918       0.532       1.584     0.7579    0.5344

  ★ p < 0.05
  † p < 0.10

  Direction-aligned interpretation:
  • ↑ marker: HR per 1 SD higher expression
  • ↓ marker: HR per 1 SD lower expression
  • Therefore, HR > 1 indicates that the predefined direction holds.


,Variable,Direction,Coefficient,HR,95% CI lower,95% CI upper,p,C-index,n,Events
0,H&E risk score,higher = risk,0.541,1.717,1.046,2.820,0.0326,0.7040,96,12
1,COMPOSITE,higher = risk,0.412,1.510,0.836,2.728,0.1716,0.6290,96,12
2,H&E + COMPOSITE,higher = risk,0.797,2.220,1.199,4.110,0.0112,0.7331,96,12
3,ARHGDIB (↓),lower = risk,0.383,1.466,0.849,2.534,0.1702,0.5899,96,12
4,GET3 (↓),lower = risk,0.471,1.601,0.873,2.938,0.1284,0.6427,96,12
5,LAP3 (↓),lower = risk,0.395,1.484,0.806,2.733,0.2050,0.6015,96,12
6,LCP1 (↓),lower = risk,0.588,1.800,1.061,3.052,0.0292,0.6956,96,12
7,PSME2 (↓),lower = risk,0.256,1.291,0.813,2.050,0.2784,0.6374,96,12
8,SERPINB9 (↓),lower = risk,0.475,1.608,0.880,2.939,0.1227,0.6131,96,12
9,TYMP (↓),lower = risk,0.103,1.109,0.638,1.929,0.7141,0.5180,96,12


In [13]:
# ── 2. Multivariate Cox ───────────────────────────────────────
print('\n' + '═'*70)
print('  2. Multivariate Cox PH')
print('═'*70)

def multi_cox(feat_dict, label):
    df = pd.DataFrame(feat_dict)
    df['T'] = y_time; df['E'] = y_event
    cph = CoxPHFitter(penalizer=0.01)
    cph.fit(df,'T','E',show_progress=False)
    print(f'\n  [{label}]')
    print(f'  AIC_partial={cph.AIC_partial_:.2f}  '
          f'log-lik={cph.log_likelihood_:.2f}  '
          f'C={cph.concordance_index_:.4f}')
    print(f'  {"Variable":18s}  {"HR":>7}  {"lo":>7}  {"hi":>7}  {"p":>8}')
    print('  '+'-'*52)
    for var in cph.params_.index:
        hr = np.exp(cph.params_[var])
        lo = np.exp(cph.confidence_intervals_.loc[var,'95% lower-bound'])
        hi = np.exp(cph.confidence_intervals_.loc[var,'95% upper-bound'])
        p  = cph.summary.loc[var,'p']
        sig= '★' if p<0.05 else ('†' if p<0.1 else ' ')
        print(f'  {var:18s}  {hr:7.3f}  {lo:7.3f}  {hi:7.3f}  {p:8.4f} {sig}')
    return cph

cph_m1   = multi_cox({'risk':risk_z},             'M1: risk only')
cph_comp = multi_cox({'composite':comp_z},         'Composite only')
cph_both = multi_cox({'risk':risk_z,'composite':comp_z}, 'risk + composite')
cph_se   = multi_cox({'SE_score':SE_score_z},      'SE_score (rank-based)')

print('\n  Model AIC_partial comparison:')
for label, cph in [('M1',cph_m1),('Composite',cph_comp),
                   ('risk+composite',cph_both),('SE_score',cph_se)]:
    print(f'    {label:20s}: AIC_partial={cph.AIC_partial_:.2f}  C={cph.concordance_index_:.4f}')


══════════════════════════════════════════════════════════════════════
  2. Multivariate Cox PH
══════════════════════════════════════════════════════════════════════

  [M1: risk only]
  AIC_partial=102.62  log-lik=-50.31  C=0.7040
  Variable                 HR       lo       hi         p
  ----------------------------------------------------
  risk                  1.717    1.046    2.820    0.0326 ★

  [Composite only]
  AIC_partial=104.97  log-lik=-51.49  C=0.6290
  Variable                 HR       lo       hi         p
  ----------------------------------------------------
  composite             1.510    0.836    2.728    0.1716  



  [risk + composite]
  AIC_partial=102.68  log-lik=-49.34  C=0.7230
  Variable                 HR       lo       hi         p
  ----------------------------------------------------
  risk                  1.735    1.044    2.884    0.0335 ★
  composite             1.504    0.833    2.716    0.1756  

  [SE_score (rank-based)]
  AIC_partial=100.05  log-lik=-49.02  C=0.7331
  Variable                 HR       lo       hi         p
  ----------------------------------------------------
  SE_score              2.220    1.199    4.110    0.0112 ★

  Model AIC_partial comparison:
    M1                  : AIC_partial=102.62  C=0.7040
    Composite           : AIC_partial=104.97  C=0.6290
    risk+composite      : AIC_partial=102.68  C=0.7230
    SE_score            : AIC_partial=100.05  C=0.7331


In [14]:
# ── 3. Log-rank test ──────────────────────────────────────────
print('\n' + '═'*70)
print('  3. Log-rank test (median split)')
print('═'*70)

split_results = {}
for name, sc in [('SE_score', SE_final), ('composite', comp_final), ('risk_score', risk)]:
    med     = np.median(sc)
    hi_mask = sc > med
    lo_mask = ~hi_mask
    lr = logrank_test(y_time[hi_mask], y_time[lo_mask],
                      event_observed_A=y_event[hi_mask],
                      event_observed_B=y_event[lo_mask])
    split_results[name] = {'lr': lr, 'hi_mask': hi_mask, 'lo_mask': lo_mask}
    sig = '★' if lr.p_value < 0.05 else ''
    print(f'  {name:15s}: p={lr.p_value:.4f} {sig}  '
          f'(high: n={hi_mask.sum()}, ev={y_event[hi_mask].sum()}  '
          f'low: n={lo_mask.sum()}, ev={y_event[lo_mask].sum()})')


══════════════════════════════════════════════════════════════════════
  3. Log-rank test (median split)
══════════════════════════════════════════════════════════════════════
  SE_score       : p=0.0294 ★  (high: n=48, ev=9  low: n=48, ev=3)
  composite      : p=0.1404   (high: n=48, ev=8  low: n=48, ev=4)
  risk_score     : p=0.1476   (high: n=48, ev=8  low: n=48, ev=4)
